In [4]:
import os
from datetime import datetime, timezone

from dotenv import load_dotenv
from supabase import create_client

In [5]:
class OrderItemService:
    def __init__(self, supabase):
        self.supabase = supabase
        self.table = "order_items"

    # Create: 주문상품 생성
    def create_order_item(
        self,
        order_id,
        product_id,
        item_name,
        item_price,
        quantity,
    ):
        data = {
            "order_id": order_id,
            "product_id": product_id,
            "item_name": item_name,
            "item_price": item_price,
            "quantity": quantity,
        }

        response = (
            self.supabase
            .table(self.table)
            .insert(data)
            .execute()
        )

        return response.data

    # Read: 특정 주문의 주문상품 전체 조회
    def get_order_items(self, order_id):
        response = (
            self.supabase
            .table(self.table)
            .select("*")
            .eq("order_id", order_id)
            .is_("deleted_at", "null")
            .execute()
        )

        return response.data

    # Read: 주문상품 한 건 조회
    def get_order_item(self, order_item_id):
        response = (
            self.supabase
            .table(self.table)
            .select("*")
            .eq("id", order_item_id)
            .is_("deleted_at", "null")
            .single()
            .execute()
        )

        return response.data

    # Update: 상품명, 가격, 수량 수정
    def update_order_item(
        self,
        order_item_id,
        item_name=None,
        item_price=None,
        quantity=None,
    ):
        data = {}

        if item_name is not None:
            data["item_name"] = item_name

        if item_price is not None:
            data["item_price"] = item_price

        if quantity is not None:
            data["quantity"] = quantity

        if not data:
            raise ValueError("수정할 데이터가 없습니다.")

        data["modified_at"] = datetime.now(timezone.utc).isoformat()

        response = (
            self.supabase
            .table(self.table)
            .update(data)
            .eq("id", order_item_id)
            .is_("deleted_at", "null")
            .execute()
        )

        return response.data

    # Delete: 소프트 삭제
    def soft_delete_order_item(self, order_item_id):
        now = datetime.now(timezone.utc).isoformat()

        response = (
            self.supabase
            .table(self.table)
            .update({
                "deleted_at": now,
                "modified_at": now,
            })
            .eq("id", order_item_id)
            .is_("deleted_at", "null")
            .execute()
        )

        return response.data

    # 삭제된 주문상품만 조회
    def get_deleted_order_items(self, order_id):
        response = (
            self.supabase
            .table(self.table)
            .select("*")
            .eq("order_id", order_id)
            .not_.is_("deleted_at", "null")
            .execute()
        )

        return response.data